## Task
Compute time intervals between adjacent commits in a chosen Git repository using **GitPython**, then visualize those intervals as a **Plotly** bar chart (newest→oldest commit pairs).

In [1]:
import sys
print(sys.executable)
print(sys.version)

C:\Users\Rainybyte\AppData\Roaming\JetBrains\DataSpell2025.3\projects\workspace\.venv\Scripts\python.exe
3.12.0 (tags/v3.12.0:0fb18b0, Oct  2 2023, 13:03:39) [MSC v.1935 64 bit (AMD64)]


In [2]:
import sys

# Install into the *current kernel* environment
!"{sys.executable}" -m pip install -U gitpython plotly pandas nbformat


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: C:\Users\Rainybyte\AppData\Roaming\JetBrains\DataSpell2025.3\projects\workspace\.venv\Scripts\python.exe -m pip install --upgrade pip


In [3]:
from pathlib import Path
from git import Repo
import pandas as pd
import plotly.express as px

In [4]:
REPO_PATH = Path(r"C:\Users\Rainybyte\Documents\Obsidian\Research Projects")

In [5]:
if REPO_PATH is None:
    REPO_PATH = Path.cwd()  # change to Path(r"C:\path\to\repo") if needed

repo = Repo(str(REPO_PATH), search_parent_directories=True)
commits = list(repo.iter_commits())  # default: HEAD history (newest -> oldest)

rows = []
for i in range(len(commits) - 1):
    newer = commits[i]
    older = commits[i + 1]
    dt_newer = pd.to_datetime(newer.committed_datetime)
    dt_older = pd.to_datetime(older.committed_datetime)
    delta = (dt_newer - dt_older).total_seconds()

    rows.append(
        {
            "pair_index": i,
            "newer_hash": newer.hexsha[:8],
            "older_hash": older.hexsha[:8],
            "newer_time": dt_newer,
            "older_time": dt_older,
            "interval_seconds": delta,
            "interval_minutes": delta / 60.0,
            "interval_hours": delta / 3600.0,
            "label": f"{newer.hexsha[:8]} → {older.hexsha[:8]}",
        }
    )

df_intervals = pd.DataFrame(rows)

df_intervals.head()

,pair_index,newer_hash,older_hash,newer_time,older_time,interval_seconds,interval_minutes,interval_hours,label
0,0,b6073e6b,8da600e0,2026-03-06 11:23:33+01:00,2026-03-06 01:12:30+01:00,36663.0,611.050000,10.184167,b6073e6b → 8da600e0
1,1,8da600e0,ff769e68,2026-03-06 01:12:30+01:00,2026-03-06 01:11:23+01:00,67.0,1.116667,0.018611,8da600e0 → ff769e68
2,2,ff769e68,d25c09e3,2026-03-06 01:11:23+01:00,2026-03-05 23:04:15+01:00,7628.0,127.133333,2.118889,ff769e68 → d25c09e3
3,3,d25c09e3,a11514e9,2026-03-05 23:04:15+01:00,2026-03-05 20:47:18+01:00,8217.0,136.950000,2.282500,d25c09e3 → a11514e9
4,4,a11514e9,f115aadc,2026-03-05 20:47:18+01:00,2026-03-05 20:46:33+01:00,45.0,0.750000,0.012500,a11514e9 → f115aadc


In [6]:
fig = px.histogram(
    df_intervals,
    x="interval_minutes",
    range_x=[0.1, 600],
    nbins=2000,  # Adjust the number of bins as needed
    hover_data={
        "newer_time": True,
        "older_time": True,
        "interval_seconds": ":.0f",
        "interval_minutes": ":.2f",
        "interval_hours": ":.3f",
    },
    title=f"Histogram of Commit Intervals ({repo.working_tree_dir})",
    labels={"interval_minutes": "Interval (minutes)"},
)

fig.update_layout(
    xaxis_title="Commit Interval (minutes)",
    yaxis_title="Frequency",
    bargap=0.1,
    height=600,
)
fig.show()